In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import time
import psutil

In [2]:
df = pd.read_csv("datasets/features_dataset.csv")

In [3]:
df.shape

(336749, 54)

In [4]:
df.columns

Index(['URL', 'Label', 'Average_Word', 'Base64_Pattern_Cnt',
       'Character_Repetition', 'Check_EXE', 'Count_-', 'Count_%', 'Count_&',
       'Count_/', 'Count_;', 'Count_?', 'Count_@', 'Count_=', 'Count_Digit',
       'Digits_Sum', 'Count_Dot', 'Count_Embed_Domain', 'Count_Letter',
       'Count_Path_Slash', 'Count_www', 'Count_Http', 'Digit/Letter',
       'Digit_Alphabet_Ratio', 'Domain_Length_Of_URL', 'Domain_URL_Ratio',
       'Double_Slash_Count', 'fd_length', 'File_Extension', 'FractalDimension',
       'FTP_Used', 'Having_Path', 'Hex_Pattern_Cnt', 'Host_Length',
       'Host_Precense_Of_Digit', 'Kolmogorov_Complexity', 'Length_Of_URL',
       'Longest_Word', 'Longest_Word_in_Hostname', 'Path_Length',
       'Port_Number', 'Having_Query', 'Query_Length', 'ShannonEntropy',
       'Short_Url', 'Special_Char_Alphabet_Ratio', 'STD_of_Words_Length',
       'Subdomain', 'Suffix', 'Tld_Length', 'Uppercase_Lowercase_Ratio',
       'URL/Path', 'use_of_ip_address', 'Vowel/Consonant'],


In [5]:
df.dtypes

URL                             object
Label                           object
Average_Word                   float64
Base64_Pattern_Cnt               int64
Character_Repetition             int64
Check_EXE                        int64
Count_-                          int64
Count_%                          int64
Count_&                          int64
Count_/                          int64
Count_;                          int64
Count_?                          int64
Count_@                          int64
Count_=                          int64
Count_Digit                      int64
Digits_Sum                       int64
Count_Dot                        int64
Count_Embed_Domain               int64
Count_Letter                     int64
Count_Path_Slash                 int64
Count_www                        int64
Count_Http                       int64
Digit/Letter                   float64
Digit_Alphabet_Ratio           float64
Domain_Length_Of_URL             int64
Domain_URL_Ratio         

In [6]:
df['Label'] = df['Label'].map({'Legitimate': 0, 'Phishing': 1})

In [7]:
df = df.drop(['URL', 'File_Extension', 'Suffix'], axis=1)

In [8]:
df = df.fillna(0)

In [9]:
print("Shape:", df.shape)
print("\nLabel dağılımı:")
print(df['Label'].value_counts())

print("\nNaN sayısı:", df.isnull().sum().sum())

print("\nObject kolonlar:")
print(df.select_dtypes(include='object').columns)

Shape: (336749, 51)

Label dağılımı:
Label
1    175806
0    160943
Name: count, dtype: int64

NaN sayısı: 0

Object kolonlar:
Index([], dtype='object')


In [11]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone

# ================= DATA =================
X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================= MODELLER =================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(),
    "Linear SVM": LinearSVC(),  # ✅ değişti
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# ================= HİBRİT MODEL =================
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
        ('svm', LinearSVC())  # ✅ burada da değişti
    ],
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

models["Stacking (Hybrid)"] = stacking_model

# ================= EĞİTİM =================
results = []
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
process = psutil.Process()

for name, model in models.items():
    try:
        print(f">>> {name} eğitiliyor...")

        current_model = clone(model)

        start_time = time.time()
        start_mem = process.memory_info().rss / (1024 ** 2)

        current_model.fit(X_train, y_train)
        train_time = time.time() - start_time

        pred_start = time.time()
        y_pred = current_model.predict(X_test)
        pred_time = time.time() - pred_start

        end_mem = process.memory_info().rss / (1024 ** 2)
        mem_used = end_mem - start_mem
        cpu_used = psutil.cpu_percent(interval=0.1)

        row = {
            "Model": name,
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred),
            "Test_Recall": recall_score(y_test, y_pred),
            "Test_F1": f1_score(y_test, y_pred),
            "Train_Time (s)": round(train_time, 4),
            "Predict_Time (s)": round(pred_time, 4),
            "Memory_Usage (MB)": round(mem_used, 2),
            "CPU_Usage (%)": cpu_used
        }

        # ROC-AUC
        try:
            if hasattr(current_model, "predict_proba"):
                y_prob = current_model.predict_proba(X_test)[:, 1]
            else:
                y_prob = current_model.decision_function(X_test)
            row["Test_ROC_AUC"] = roc_auc_score(y_test, y_prob)
        except:
            row["Test_ROC_AUC"] = None

        print(f"    {name} için Cross-Validation...")
        cv_scores = cross_validate(clone(model), X, y, cv=5, scoring=scoring, n_jobs=-1)

        row["CV_Accuracy"] = cv_scores["test_accuracy"].mean()
        row["CV_F1"] = cv_scores["test_f1"].mean()
        row["CV_ROC_AUC"] = cv_scores["test_roc_auc"].mean()

        results.append(row)
        print(f"--- {name} tamamlandı ---\n")

    except Exception as e:
        print(f"!!! {name} hatası: {e}")

# ================= SONUÇ =================
results_table = pd.DataFrame(results).sort_values("Test_Accuracy", ascending=False)
results_table = results_table.reset_index(drop=True)
results_table.index = results_table.index + 1

print("\n=== TÜM MODELLER PERFORMANS TABLOSU ===")
print(results_table)

>>> Logistic Regression eğitiliyor...
    Logistic Regression için Cross-Validation...
--- Logistic Regression tamamlandı ---

>>> Random Forest eğitiliyor...
    Random Forest için Cross-Validation...
--- Random Forest tamamlandı ---

>>> Decision Tree eğitiliyor...
    Decision Tree için Cross-Validation...
--- Decision Tree tamamlandı ---

>>> Linear SVM eğitiliyor...
    Linear SVM için Cross-Validation...
--- Linear SVM tamamlandı ---

>>> Naive Bayes eğitiliyor...
    Naive Bayes için Cross-Validation...
--- Naive Bayes tamamlandı ---

>>> KNN eğitiliyor...
    KNN için Cross-Validation...
--- KNN tamamlandı ---

>>> Gradient Boosting eğitiliyor...
    Gradient Boosting için Cross-Validation...
--- Gradient Boosting tamamlandı ---

>>> XGBoost eğitiliyor...
    XGBoost için Cross-Validation...
--- XGBoost tamamlandı ---

>>> Stacking (Hybrid) eğitiliyor...


[16:17:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.



    Stacking (Hybrid) için Cross-Validation...


[16:17:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:17:58] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:18:14] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:18:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[16:18:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" }

--- Stacking (Hybrid) tamamlandı ---


=== TÜM MODELLER PERFORMANS TABLOSU ===
                 Model  Test_Accuracy  Test_Precision  Test_Recall   Test_F1  \
1              XGBoost       0.997803        0.999629     0.996161  0.997892   
2    Stacking (Hybrid)       0.997788        0.999515     0.996246  0.997878   
3        Random Forest       0.997535        0.999116     0.996161  0.997636   
4  Logistic Regression       0.997149        0.999514     0.995023  0.997264   
5        Decision Tree       0.996897        0.998034     0.996018  0.997025   
6    Gradient Boosting       0.996184        0.999513     0.993174  0.996334   
7                  KNN       0.991656        0.996470     0.987515  0.991972   
8           Linear SVM       0.964959        0.992168     0.940303  0.965539   
9          Naive Bayes       0.921292        0.993423     0.854896  0.918968   

   Train_Time (s)  Predict_Time (s)  Memory_Usage (MB)  CPU_Usage (%)  \
1          0.5496            0.0146            